# A-share Barra risk model — validation, 2026-11-11

Plan 05 phase 5/6. The CNE5-style model of `config/risk_model.yaml` (`cne5_v3`: country + 31 Shenwan L1 industries +
ten styles — Size, Beta, Momentum, Residual Volatility, Non-linear Size, Book-to-Price, Liquidity, Earnings Yield,
Growth, Leverage, plus S&P 500 sensitivity, USD sensitivity and cash-flow yield; analyst descriptors not built, weights renormalised per RISK_FACTOR_LIST §1; D-128, D-131, D-134). Frames come from
`make risk-model` (derived/risk_*, D-127); this notebook only reads them and the validation report through the package.

Acceptance (plan 05 ph5): random-portfolio bias statistics within [0.85, 1.15] over the last 12 months; median
daily R² ≥ 0.25. A bias statistic is sd(r_{t+1} / σ_t): 1 = unbiased, > 1 = risk under-forecast; the 95% band
for one series of 252 forecasts is about 1 ± 0.09.

In [1]:
import numpy as np
import pandas as pd

from quant_cn.core.config import Config
from quant_cn.core.frames import FACTOR_COV, FACTOR_RETURN, RISK_VALIDATION
from quant_cn.lake.lake_catalog import LakeCatalog
from quant_cn.lake.lake_query import LakeQuery
from quant_cn.risk.risk_decomposer import RiskDecomposer
from quant_cn.risk.risk_model_factory import RiskModelFactory
from quant_cn.visualization.chart_theme import ChartTheme
from quant_cn.visualization.risk_chart import RiskChart

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.4f}".format)
config = Config.load()
settings = config.risk_model
catalog = LakeCatalog(config.lake.root, config.datasets, read_only=True)
query = LakeQuery(catalog)
model = RiskModelFactory(query).build(settings)
report = RISK_VALIDATION.validate(
    pd.read_parquet(config.lake.root / "derived" / "reports" / f"risk_{settings.name}" / "validation.parquet")
)
factor_returns = FACTOR_RETURN.validate(query.sql("SELECT * FROM derived.risk_factor_return ORDER BY trade_date, factor"))
last = factor_returns["trade_date"].max()
theme = ChartTheme.from_mode("light")
print(f"model {settings.name}: {settings.style_names}")
print(f"factor returns {factor_returns['trade_date'].min()} -> {last}, {factor_returns['factor'].nunique()} factors")
print(report["notes"].iloc[0])

model cne5_v3: ['size', 'beta', 'momentum', 'residual_volatility', 'non_linear_size', 'book_to_price', 'liquidity', 'earnings_yield', 'growth', 'leverage', 'us_sensitivity', 'usd_sensitivity', 'cash_flow_yield']
factor returns 20190103 -> 20261111, 45 factors
run 5b2dcc3a3c7f; model cne5_v3


## 1. Acceptance

In [2]:
summary = report[report["scope"].isin(["model", "window", "random_portfolios"])]
display(summary[["metric", "scope", "value", "window"]])
r2 = summary.set_index(["metric", "scope"])["value"]
share = r2[("share_in_band", "random_portfolios")]
print(f"median R2 (all history) {r2[('median_r2', 'model')]:.3f} -> {'PASS' if r2[('median_r2', 'model')] >= 0.25 else 'FAIL'} (>= 0.25)")
print(f"random portfolios in [0.85, 1.15]: {share:.0%}, median bias {r2[('median_bias', 'random_portfolios')]:.3f}")

,metric,scope,value,window
0,median_r2,model,0.2807,20190103-20261110
1,mean_r2,model,0.3183,20190103-20261110
2,median_adj_r2,model,0.2732,20190103-20261110
3,median_r2,window,0.3073,20251028-20261110
4,mean_r2,window,0.3348,20251028-20261110
5,median_adj_r2,window,0.3012,20251028-20261110
450,share_in_band,random_portfolios,1.0000,20251028-20261110
451,median_bias,random_portfolios,1.0177,20251028-20261110


median R2 (all history) 0.281 -> PASS (>= 0.25)
random portfolios in [0.85, 1.15]: 100%, median bias 1.018


## 2. Factor bias statistics (last 12 months)
Style factors and the country factor first; industries summarised.

In [3]:
bias = report[(report["metric"] == "bias") & report["scope"].str.startswith("factor:")].copy()
bias["factor"] = bias["scope"].str[len("factor:"):]
styles = bias[~bias["factor"].str.startswith("ind_")].set_index("factor")["value"].sort_values()
display(styles.to_frame("bias"))
ind = bias[bias["factor"].str.startswith("ind_")]["value"]
print(f"industries: median {ind.median():.3f}, range {ind.min():.2f}-{ind.max():.2f}, in [0.8, 1.2]: {ind.between(0.8, 1.2).mean():.0%}")
bench = report[(report["metric"] == "bias") & report["scope"].str.endswith((".SH", ".SZ"))]
display(bench[["scope", "value"]].rename(columns={"scope": "benchmark", "value": "bias"}))

,bias
factor,
non_linear_size,0.8957
us_sensitivity,0.9318
leverage,0.9486
residual_volatility,0.9704
cash_flow_yield,0.9890
growth,1.0094
earnings_yield,1.0219
book_to_price,1.0487
size,1.0502


industries: median 0.990, range 0.85-1.25, in [0.8, 1.2]: 94%


,benchmark,bias
141,000300.SH,0.9341
144,000852.SH,1.0455
147,000905.SH,1.0820


## 3. Explanatory power and factor returns

In [4]:
r2_daily = factor_returns.drop_duplicates("trade_date").set_index("trade_date")["r2"]
by_year = r2_daily.groupby(r2_daily.index.str[:4]).agg(["median", "mean", "count"])
display(by_year)
style_stats = factor_returns[~factor_returns["factor"].str.startswith("ind_")].groupby("factor").agg(
    ann_ret=("ret", lambda r: r.mean() * 252), ann_vol=("ret", lambda r: r.std() * np.sqrt(252)),
    mean_abs_t=("t_stat", lambda t: t.abs().mean()), share_abs_t_gt_2=("t_stat", lambda t: (t.abs() > 2).mean()),
)
display(style_stats.sort_values("ann_vol", ascending=False))

,median,mean,count
trade_date,,,
2019,0.2279,0.3016,243
2020,0.2520,0.3069,243
2021,0.2394,0.2647,243
2022,0.3074,0.3486,242
2023,0.2890,0.3080,242
2024,0.3593,0.3982,242
2025,0.2351,0.2681,243
2026,0.3283,0.3578,206


,ann_ret,ann_vol,mean_abs_t,share_abs_t_gt_2
factor,,,,
country,0.0841,0.1866,12.9929,0.8845
beta,0.0877,0.0625,5.0151,0.7201
momentum,0.0519,0.0479,4.5323,0.7117
non_linear_size,-0.0586,0.0425,3.6194,0.5935
size,-0.0439,0.0422,5.2745,0.7348
residual_volatility,-0.0617,0.0404,3.1296,0.5893
liquidity,-0.0848,0.0365,2.8735,0.5646
book_to_price,-0.0022,0.0235,2.2303,0.4695
us_sensitivity,0.0149,0.0223,2.5376,0.5095


In [5]:
RiskChart(theme, "cumulative").render(factor_returns)

## 4. Factor correlations on the last session

In [6]:
cov = FACTOR_COV.validate(query.sql("SELECT * FROM derived.risk_factor_cov WHERE trade_date = ?", [last]))
RiskChart(theme, "correlation", trade_date=last).render(cov)

## 5. Attribution: CSI 300, and CSI 500 active against CSI 300
Latest index_weight weights on or before the last session; daily units annualised by √252 in the charts.

In [7]:
weights = query.sql(
    "SELECT index_code, con_code AS ts_code, weight / 100 AS weight FROM curated.index_weight "
    "WHERE index_code IN ('000300.SH', '000905.SH') AND trade_date <= ? "
    "QUALIFY trade_date = max(trade_date) OVER (PARTITION BY index_code)", [last])
csi300 = weights[weights["index_code"] == "000300.SH"].set_index("ts_code")["weight"]
csi500 = weights[weights["index_code"] == "000905.SH"].set_index("ts_code")["weight"]
snapshot = model.load_snapshot(last)
decomp = pd.concat([
    RiskDecomposer().compute(snapshot, csi300, "CSI 300"),
    RiskDecomposer().compute(snapshot, csi500, "CSI 500 vs 300", benchmark=csi300),
])
top = decomp[decomp["component"].isin(["total", "factor", "specific", "active_total", "active_factor", "active_specific"])]
display(top.assign(annualised=top["risk"] * np.sqrt(252))[["portfolio", "component", "annualised", "pct"]])
RiskChart(theme, "decomposition", portfolio="CSI 300").render(decomp)

,portfolio,component,annualised,pct
45,CSI 300,factor,0.1592,0.9702
46,CSI 300,specific,0.0279,0.0298
47,CSI 300,total,0.1616,1.0000
45,CSI 500 vs 300,active_factor,0.1437,0.9409
46,CSI 500 vs 300,active_specific,0.0360,0.0591
47,CSI 500 vs 300,active_total,0.1482,1.0000
138,CSI 500 vs 300,factor,0.2406,0.9911
139,CSI 500 vs 300,specific,0.0228,0.0089
140,CSI 500 vs 300,total,0.2417,1.0000


In [8]:
exposures = decomp[(decomp["portfolio"] == "CSI 500 vs 300") & decomp["component"].str.startswith("active_x:")]
active = exposures.assign(factor=exposures["component"].str[len("active_x:"):]).set_index("factor")["risk"]
display(active[~active.index.str.startswith("ind_")].sort_values().to_frame("active exposure"))

,active exposure
factor,
size,-2.3997
earnings_yield,-0.9615
leverage,-0.4166
book_to_price,-0.3443
cash_flow_yield,-0.2236
growth,-0.1401
residual_volatility,-0.0362
momentum,-0.0082
country,-0.0044


## 6. Monitor: AI / 数据中心 (AIDC) sensitivity — shown separately, not in the risk model
Style `ai_dc_sensitivity` = partial beta on the circ_mv-weighted 885887.TI 数据中心(AIDC) basket (excluding the
stock, from the 2020-03-06 list date), China market controlled (D-130). Its daily return is estimated by adding it
to each session's regression; the risk model's covariance and forecasts do not include it. Membership is a 2026
snapshot, so returns before 2025 carry hindsight.

In [9]:
monitor_path = config.lake.root / "derived" / "reports" / f"risk_{settings.name}" / "monitor_returns.parquet"
monitor = FACTOR_RETURN.validate(pd.read_parquet(monitor_path)) if monitor_path.exists() else factor_returns.iloc[:0]
aidc = monitor[monitor["factor"] == "ai_dc_sensitivity"].set_index("trade_date").sort_index()
by_year = aidc.groupby(aidc.index.str[:4]).agg(
    ret=("ret", lambda r: (1 + r).prod() - 1), ann_vol=("ret", lambda r: r.std() * np.sqrt(252)),
    mean_abs_t=("t_stat", lambda t: t.abs().mean()), r2_gain=("r2", "mean"), sessions=("ret", "size"))
display(by_year)
roll = pd.DataFrame({"ann_vol_60d": aidc["ret"].rolling(60).std() * np.sqrt(252),
                     "abs_t_60d": aidc["t_stat"].abs().rolling(60).mean()})
display(roll.tail(1))
RiskChart(theme, "cumulative", factors=["ai_dc_sensitivity"]).render(monitor)

,ret,ann_vol,mean_abs_t,r2_gain,sessions
trade_date,,,,,
2020,-0.0009,0.0392,2.9087,0.0026,140
2021,0.0897,0.0516,4.1062,0.0047,243
2022,0.0105,0.0726,6.2601,0.0078,242
2023,0.0999,0.0616,5.7803,0.0065,242
2024,0.0337,0.0672,4.8216,0.0043,242
2025,0.1326,0.0648,5.3584,0.0058,243
2026,0.1563,0.0882,6.5346,0.0080,206


,ann_vol_60d,abs_t_60d
trade_date,,
20261111,0.0919,5.9316


In [10]:
ex = query.sql(
    "SELECT e.ts_code, e.exposure, i.l1_name, s.name FROM derived.risk_exposure e "
    "JOIN curated.stock_basic s USING (ts_code) "
    "LEFT JOIN derived.industry_l1 i ON i.ts_code = e.ts_code AND coalesce(i.valid_from, '0') <= e.trade_date "
    "AND coalesce(i.valid_to, '99991231') >= e.trade_date "
    "WHERE e.trade_date = ? AND e.factor = 'ai_dc_sensitivity'", [last]).drop_duplicates("ts_code")
display(ex.groupby("l1_name")["exposure"].agg(["mean", "size"]).sort_values("mean", ascending=False).round(2))
display(pd.concat([ex.nlargest(20, "exposure"), ex.nsmallest(20, "exposure")])[["ts_code", "name", "l1_name", "exposure"]])

,mean,size
l1_name,,
电子,1.5900,475
通信,1.0600,115
机械设备,0.5200,514
国防军工,0.1100,133
汽车,0.0900,283
电力设备,0.0900,362
计算机,0.0100,316
家用电器,-0.0600,91
有色金属,-0.0800,138


,ts_code,name,l1_name,exposure
263,000811.SZ,冰轮环境,机械设备,3.0033
482,001389.SZ,广合科技,电子,3.0033
726,002281.SZ,光迅科技,通信,3.0033
832,002384.SZ,东山精密,电子,3.0033
902,002436.SZ,兴森科技,电子,3.0033
917,002463.SZ,沪电股份,电子,3.0033
932,002484.SZ,江海股份,电子,3.0033
1231,002851.SZ,麦格米特,电力设备,3.0033
1280,002916.SZ,深南电路,电子,3.0033
1314,002938.SZ,鹏鼎控股,电子,3.0033


## 7. Reading the result
- R² by year and the style t-statistics show which styles carry daily return variance in A-shares.
- Factors with bias > 1.15 have their risk under-forecast in the last year (regime shifts faster than the 90-day
  volatility half-life); the volatility regime adjustment lifts the whole matrix, not single factors.
- Next (plan 06): the optimiser consumes FACTOR_COV / SPECIFIC_RISK; `factor` may neutralise on RISK_EXPOSURE.

In [11]:
catalog.close()